# Đánh giá KNN tốt nhất

GridSearchCV thử 16 tổ hợp với Stratified CV 5 fold và chọn theo F1-macro.
Pipeline gồm StandardScaler và KNeighborsClassifier; scaler được fit riêng trên train của từng fold. Tập test chỉ dùng để đánh giá cuối cùng.

Lưới tham số: `knn__n_neighbors = [3, 5, 11, 21]`, `knn__weights = ["uniform", "distance"]`, `knn__p = [1, 2]`.
`p=1` dùng khoảng cách Manhattan; `p=2` dùng khoảng cách Euclidean.

## 1. Thư viện và đường dẫn

In [1]:
import json
from pathlib import Path
import sys

import joblib
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

project_dir = Path.cwd().resolve()
if project_dir.name == "knn":
    project_dir = project_dir.parents[1]
elif project_dir.name == "models":
    project_dir = project_dir.parent
sys.path.insert(0, str(project_dir))

from models.common import read_split, timestamp, write_json
from models.reporting import SUMMARY_COLUMNS

model_dir = project_dir / "models" / "knn"
pd.set_option("display.max_columns", None)

## 2. Đọc model và dữ liệu

In [2]:
artifact = joblib.load(model_dir / "training_results.joblib")
search = artifact["search"]
model = joblib.load(model_dir / "model.joblib")
X_train, y_train = read_split("train")
X_test, y_test = read_split("test")
classes = model.classes_.tolist()
results = search.cv_results_
best_index = int(search.best_index_)

## 3. Siêu tham số tốt nhất

In [3]:
display(pd.DataFrame([search.best_params_]))

,knn__n_neighbors,knn__p,knn__weights
0,5,1,distance


## 4. Bảng tổng hợp

In [4]:
train_prediction = model.predict(X_train)
prediction = model.predict(X_test)

options = {"labels": classes, "average": "macro", "zero_division": 0}
summary = pd.DataFrame([{
    "Model": "KNN", "Configuration": "GridSearch",
    "CV_accuracy_mean": float(results["mean_test_accuracy"][best_index]),
    "CV_f1_macro_mean": float(results["mean_test_f1_macro"][best_index]),
    "CV_f1_macro_std": float(results["std_test_f1_macro"][best_index]),
    "Test_accuracy": float(accuracy_score(y_test, prediction)),
    "Test_precision_macro": float(precision_score(y_test, prediction, **options)),
    "Test_recall_macro": float(recall_score(y_test, prediction, **options)),
    "Test_f1_macro": float(f1_score(y_test, prediction, **options)),
    "Train_f1_macro": float(f1_score(y_train, train_prediction, **options)),
    "CV_train_validation_gap_F1": float(
        results["mean_train_f1_macro"][best_index] - results["mean_test_f1_macro"][best_index]
    ),
    "Preprocessing": "StandardScaler",
    "Best_grid_parameters": json.dumps(search.best_params_, sort_keys=True),
}], columns=SUMMARY_COLUMNS)
display(summary)

,Model,Configuration,CV_accuracy_mean,CV_f1_macro_mean,CV_f1_macro_std,Test_accuracy,Test_precision_macro,Test_recall_macro,Test_f1_macro,Train_f1_macro,CV_train_validation_gap_F1,Preprocessing,Best_grid_parameters
0,KNN,GridSearch,0.975568,0.97544,0.01126,0.981818,0.982348,0.981818,0.98169,1.0,0.02456,StandardScaler,"{""knn__n_neighbors"": 5, ""knn__p"": 1, ""knn__wei..."


## 5. Kết quả GridSearch

In [5]:
grid_rows = []
for i, parameters in enumerate(results["params"]):
    grid_rows.append({
        "Model": "KNN", "Candidate": i + 1,
        "Parameters": json.dumps(parameters, sort_keys=True),
        "Selected": i == best_index,
        "CV_accuracy_mean": float(results["mean_test_accuracy"][i]),
        "CV_f1_macro_mean": float(results["mean_test_f1_macro"][i]),
        "CV_f1_macro_std": float(results["std_test_f1_macro"][i]),
    })
grid = pd.DataFrame(grid_rows)
display(grid)

,Model,Candidate,Parameters,Selected,CV_accuracy_mean,CV_f1_macro_mean,CV_f1_macro_std
0,KNN,1,"{""knn__n_neighbors"": 3, ""knn__p"": 1, ""knn__wei...",False,0.972727,0.972640,0.011956
1,KNN,2,"{""knn__n_neighbors"": 3, ""knn__p"": 1, ""knn__wei...",False,0.973295,0.973192,0.011833
2,KNN,3,"{""knn__n_neighbors"": 3, ""knn__p"": 2, ""knn__wei...",False,0.964205,0.964003,0.008834
3,KNN,4,"{""knn__n_neighbors"": 3, ""knn__p"": 2, ""knn__wei...",False,0.967614,0.967432,0.009237
4,KNN,5,"{""knn__n_neighbors"": 5, ""knn__p"": 1, ""knn__wei...",False,0.973295,0.973046,0.010786
5,KNN,6,"{""knn__n_neighbors"": 5, ""knn__p"": 1, ""knn__wei...",True,0.975568,0.975440,0.011260
6,KNN,7,"{""knn__n_neighbors"": 5, ""knn__p"": 2, ""knn__wei...",False,0.965341,0.965011,0.012439
7,KNN,8,"{""knn__n_neighbors"": 5, ""knn__p"": 2, ""knn__wei...",False,0.968182,0.967952,0.012081
8,KNN,9,"{""knn__n_neighbors"": 11, ""knn__p"": 1, ""knn__we...",False,0.966477,0.966166,0.011380
9,KNN,10,"{""knn__n_neighbors"": 11, ""knn__p"": 1, ""knn__we...",False,0.969886,0.969681,0.009548


## 6. Lưu kết quả đánh giá

In [7]:
evaluated_at = timestamp()
evaluation = {
    "model_name": "KNN", "evaluated_at": evaluated_at,
    "tables": {
        "Tong_hop": summary.to_dict(orient="records"),
        "GridSearch": grid.to_dict(orient="records"),
    },
}
selected = summary.iloc[0]
metadata = {
    "model_file": "model.joblib", "model_name": "KNN",
    "estimator": "KNeighborsClassifier", "configuration": "GridSearch",
    "parameters": search.best_params_, "preprocessing": "StandardScaler",
    "input_format": "Unscaled feature values in the order listed in features",
    "features": X_train.columns.tolist(), "classes": classes,
    "selection_metric": "CV_f1_macro_mean",
    "cv_f1_macro_mean": float(selected["CV_f1_macro_mean"]),
    "cv_f1_macro_std": float(selected["CV_f1_macro_std"]),
    "train_samples": len(X_train), "test_samples": len(X_test),
    "n_splits": search.n_splits_, "cv_method": str(search.cv),
    "training_finished_at": artifact["finished_at"],
    "versions": artifact["versions"], "evaluated_at": evaluated_at,
    "test_metrics": {
        "accuracy": float(selected["Test_accuracy"]),
        "precision_macro": float(selected["Test_precision_macro"]),
        "recall_macro": float(selected["Test_recall_macro"]),
        "f1_macro": float(selected["Test_f1_macro"]),
    },
}
write_json(model_dir / "model_metadata.json", metadata)
write_json(model_dir / "evaluation_results.json", evaluation)